# Modelling Credit Card Fraud

This notebook continues from the Exploratory Data Analysis (EDA) done in [`dataCollections.ipynb`](dataCollections.ipynb). In this part, we wish to use machine learning techniques to extract valuable insights from the data and apply various models to see which ones perform the best at predicting fraud.

## Setup
First, we reload the data and the corresponding libraries needed along with initializing a random seed.


In [1]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler

seed = 31415
np.random.seed(seed)

path = kagglehub.dataset_download('whenamancodes/fraud-detection')
df = pd.read_csv(path + '/creditcard.csv')

C:\Users\Faded\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preproccessing

In the preprocessing stage, we wish to transform the amounts field to an appropriate range. This is done since features V1-V28 are obtained by PCA.

In [9]:
df_t = df.copy()

# Robust scaler since we have many outliers.
scaler = RobustScaler()

df_t['Amount_scaled'] = scaler.fit_transform(df_t[['Amount']])

df_t['Hour'] = (df_t['Time'] % 86400) / 3600

# Get rid of redundant
df_t = df_t.drop(columns=['Time','Amount'])

## Train/Test Split

Next we split the data into 80% training data and 20% test data. This ensures that we can adequately train our models while leaving a large enough test sample to evaluate the models.

In [12]:
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

In [14]:
print("df_t columns:", df_t.columns.tolist())
print("df columns:", df.columns.tolist())

df_t columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Amount_scaled', 'Hour']
df columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [15]:
X = df_t.drop('Class', axis=1)
y = df_t['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = RobustScaler()
X_train[['Amount_scaled', 'Hour']] = scaler.fit_transform(X_train[['Amount_scaled', 'Hour']])
X_test[['Amount_scaled', 'Hour']] = scaler.transform(X_test[['Amount_scaled', 'Hour']])

In [16]:
print(f"Train: {X_train.shape}, fraud rate: {y_train.mean()*100:.3f}%")
print(f"Test:  {X_test.shape}, fraud rate: {y_test.mean()*100:.3f}%")
print("\nScaled columns:")
print(X_train[['Amount_scaled', 'Hour']].describe())

Train: (227845, 30), fraud rate: 0.173%
Test:  (56962, 30), fraud rate: 0.172%

Scaled columns:
       Amount_scaled           Hour
count  227845.000000  227845.000000
mean        0.921034      -0.053812
std         3.489528       0.669353
min        -0.306193      -1.719978
25%        -0.227697      -0.505204
50%         0.000000       0.000000
75%         0.772303       0.494796
max       357.260404       1.029886
